# HydroPulse: train the learned model (Colab)
**What this does.** The HydroPulse engine already predicts how many days a filter has left using physics. This notebook trains a small neural network on top of the engine's physics features. It outputs a **calibrated range of days left** (10%, 50%, 90%) and the **probability that the carbon filter is spent**. The result is one small file, `model.json`, that the app loads automatically.

**Time:** about 25 to 40 minutes in total. **Hardware:** the model is tiny, so a normal CPU runtime is enough. A free **GPU** speeds it a little (Runtime > Change runtime type > T4 GPU). A TPU is **not needed** and not worth the setup.

**Before you start (once):**
1. In Google Drive, create a folder named `atomquest` (My Drive > atomquest).
2. Upload `HydroPulse_Final_Prototype.zip` into that folder.
3. Run the cells below from top to bottom. Everything is saved back into `atomquest/`.

**Honest note.** All training data comes from the HydroPulse simulator, with sensor noise copied from a real, public 540-day plant record. The model learns the simulator's behaviour. It must be re-trained on real purifier logs before anyone claims real-world accuracy.

In [ ]:
# 1. Connect Google Drive and create the folders
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, urllib.request, zipfile, glob, time
BASE = '/content/drive/MyDrive/atomquest'
for d in ('data', 'models', 'code'): os.makedirs(f'{BASE}/{d}', exist_ok=True)
print('Drive folder ready:', os.listdir(BASE))

In [ ]:
# 2. Unpack the project from Drive
z = glob.glob(f'{BASE}/**/HydroPulse_Final_Prototype.zip', recursive=True)
assert z, 'Upload HydroPulse_Final_Prototype.zip into My Drive/atomquest/ first, then re-run this cell.'
zipfile.ZipFile(z[0]).extractall(f'{BASE}/code')
HP = f'{BASE}/code/hp'; sys.path.insert(0, HP); os.chdir(HP)
print(sorted(os.listdir(HP)))

## 3. Dataset (real, public, free)
**Real 540-day record of an operating reverse-osmosis plant** (daily RO product volume). Repository page: https://github.com/Asaeel/ro-fouling-early-warning
Direct file: https://raw.githubusercontent.com/Asaeel/ro-fouling-early-warning/main/data/MBR_RO_AIOT_daily_flow_540d.xlsx

We use it to measure **real sensor noise and outage patterns**, which then drive the simulator. It has flow only (no salinity, temperature or fouling labels), so it cannot teach lifetime prediction by itself.

*Optional second source:* the open RO supervisory dataset on Zenodo, https://zenodo.org/records/21906111 (CC-BY 4.0, flow, pressure, temperature, pH, conductivity). Download it by hand into `atomquest/data/` and test it with `import_csv.py`.

In [ ]:
# 3. Download the real dataset into Drive and refresh the real-noise file
import statistics as st
URL = 'https://raw.githubusercontent.com/Asaeel/ro-fouling-early-warning/main/data/MBR_RO_AIOT_daily_flow_540d.xlsx'
dst = f'{BASE}/data/MBR_RO_AIOT_daily_flow_540d.xlsx'
if not os.path.exists(dst): urllib.request.urlretrieve(URL, dst)
%pip install -q openpyxl
import openpyxl
ws = openpyxl.load_workbook(dst, read_only=True, data_only=True).worksheets[0]
rows = [r for r in ws.iter_rows(min_row=3, values_only=True) if isinstance(r[0], int)]
q = [r[4] for r in rows]; res = []
for i in range(7, len(q) - 7):
    m = st.median(q[i-7:i+8])
    if m > 0 and q[i] > .3 * m: res.append(round(q[i] / m - 1, 4))
json.dump(dict(rel_resid=res), open(f'{HP}/data/real_noise.json', 'w'))
print(len(rows), 'days read;', len(res), 'real noise samples saved')

In [ ]:
# 4. Build training data from the simulator (physics-only estimator, no model yet)
import numpy as np
import engine, evaluate as ev
engine.MODEL = None
TRAIN_SEEDS, VAL_SEEDS = range(1000, 1024), range(1030, 1036)     # the benchmark uses seed 7: never seen in training
def gen(seeds):
    X, Y, C, M = [], [], [], []
    for sd in seeds:
        for law, real in (('linear', False), ('harder', False), ('linear', True)):
            H = ev.make_env(sd, real); _, info = ev.simulate(H, law, 'none', 0, True)
            for inf in info:
                T = inf['fail']
                for d, f, c in inf['feat']:
                    ok = T is not None and d < T and f[-1] * 100 < 150           # f[-1] is the physics days-left / 100
                    X.append(f); C.append(c); Y.append(np.log1p(T - d) if ok else 0.); M.append(1. if ok else 0.)
    return [np.array(a, 'float32') for a in (X, Y, C, M)]
t0 = time.time(); tr = gen(TRAIN_SEEDS); print('train rows', len(tr[0]), round(time.time() - t0), 's')
va = gen(VAL_SEEDS); print('validation rows', len(va[0]))

In [ ]:
# 5. Train a small network: 3 range outputs (pinball loss) + 1 carbon-spent output
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', dev)
mu, sd = tr[0].mean(0), tr[0].std(0) + 1e-6
P = lambda D: [torch.tensor((D[0] - mu) / sd).to(dev)] + [torch.tensor(a).to(dev) for a in D[1:]]
Tt, Vv = P(tr), P(va)
net = nn.Sequential(nn.Linear(Tt[0].shape[1], 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 4)).to(dev)
qs = torch.tensor([.1, .5, .9], device=dev)
def loss(o, y, c, m):
    e = y[:, None] - o[:, :3]; pin = torch.maximum(qs * e, (qs - 1) * e).mean(1)
    return (pin * m).sum() / m.sum().clamp(min=1) + nn.functional.binary_cross_entropy_with_logits(o[:, 3], c)
opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=1e-4); best = 1e9
for ep in range(60):
    perm = torch.randperm(len(Tt[0]), device=dev); net.train()
    for i in range(0, len(perm), 4096):
        b = perm[i:i+4096]; opt.zero_grad(); loss(net(Tt[0][b]), Tt[1][b], Tt[2][b], Tt[3][b]).backward(); opt.step()
    net.eval()
    with torch.no_grad(): v = loss(net(Vv[0]), *Vv[1:]).item()
    if v < best: best, state = v, {k: x.clone() for k, x in net.state_dict().items()}
    if ep % 10 == 0: print('epoch', ep, 'validation loss', round(v, 4))
net.load_state_dict(state); print('best validation loss', round(best, 4))

In [ ]:
# 6. Calibrate the range on the validation seeds, then check it
with torch.no_grad(): o = net(Vv[0]).cpu().numpy()
ok = Vv[3].cpu().numpy() > 0; y = Vv[1].cpu().numpy()
off_lo = float(np.quantile(y[ok] - o[ok, 0], .10)); off_hi = float(np.quantile(y[ok] - o[ok, 2], .90))
lo, hi = o[ok, 0] + off_lo, o[ok, 2] + off_hi
print('range holds the truth', round(100 * np.mean((y[ok] >= lo) & (y[ok] <= hi))), '% of the time (target 80%)')
print('truth never earlier than the safe date', round(100 * np.mean(y[ok] >= lo)), '% (target 90%)')
from sklearn.metrics import roc_auc_score
print('carbon-spent detection AUC', round(roc_auc_score(Vv[2].cpu().numpy(), o[:, 3]), 3), '(0.5 = guessing, 1.0 = perfect)')

In [ ]:
# 7. Export to model.json (pure numbers, no ML libraries needed to run it) and save to Drive
W = [dict(W=m.weight.detach().cpu().numpy().tolist(), b=m.bias.detach().cpu().numpy().tolist()) for m in net if isinstance(m, nn.Linear)]
model = dict(mean=mu.tolist(), std=sd.tolist(), layers=W, off=[off_lo, off_hi])
json.dump(model, open(f'{BASE}/models/model.json', 'w')); json.dump(model, open(f'{HP}/model.json', 'w'))
print('saved to', f'{BASE}/models/model.json', round(os.path.getsize(f'{HP}/model.json') / 1024), 'KB')

In [ ]:
# 8. Honest comparison on the untouched test fleet (seed 7): physics only vs physics + learned model
def run(args, out):
    r = subprocess.run([sys.executable, 'evaluate.py'] + args + ['--out', out], capture_output=True, text=True, cwd=HP); print(r.stderr[-500:]); return json.load(open(f'{HP}/{out}'))
phys = run(['--no-model'], 'results_physics.json'); learn = run([], 'results_learned.json')
print(f"{'measure':58s}{'physics':>22s}{'learned':>22s}   (normal / harder / real-noise)")
for a, b in zip(phys['rows'], learn['rows']): print(f"{a['name'][:56]:58s}{str(a['vals']):>22s}{str(b['vals']):>22s}")

In [ ]:
# 9. Decide. Only adopt the learned model if the table above shows it is better where it matters.
ADOPT = False          # change to True to make the app show the learned results
import shutil
if ADOPT: shutil.copy(f'{HP}/results_learned.json', f'{HP}/results.json'); shutil.copy(f'{HP}/results_learned.md', f'{HP}/results.md')
else:
    os.remove(f'{HP}/model.json'); print('Learned model NOT adopted: removed model.json from the project folder (a copy stays in Drive/models).')
out = f'{BASE}/HydroPulse_Final_Prototype_trained.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(HP):
        for f in files:
            if '__pycache__' not in root: zf.write(os.path.join(root, f), os.path.join('hp', os.path.relpath(os.path.join(root, f), HP)))
print('Download this file from Drive:', out)